# micrograd 3: `backward()`

Karpathy, *Zero to Hero* #1, from ~1:09 to ~1:27 (stop when he starts breaking `tanh` into `exp`; that's the stretch block, or session 4).

Goal: make the graph compute every gradient by itself. Last session you were the backprop algorithm; today you write it down so Python can be.

**Rules:** pause before each step he does and do it first. Type everything. Predict before you run.

**Your own question from last time gets answered today:** what happens to a gradient when one node feeds into two places?

## Block 0: carry-over (5 min)
Your `Value` from session 2, unchanged, and the tanh neuron. `draw_dot` now lives in `viz.py`.

Predict before running: what will every `grad` box show, and why isn't it the hand values from last time?

In [ ]:
import math
from viz import draw_dot

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self.grad = 0.0

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        return out

    def __radd__(self, other):   # handles other + self
        return self + other

    def __rmul__(self, other):   # handles other * self
        return self * other

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        return out


def neuron():
    # Fresh objects every call, so old grads never leak between experiments.
    x1 = Value(2.0, label='x1'); x2 = Value(0.0, label='x2')
    w1 = Value(-3.0, label='w1'); w2 = Value(1.0, label='w2')
    b = Value(6.8813735870195432, label='b')
    x1w1 = x1*w1; x1w1.label = 'x1w1'
    x2w2 = x2*w2; x2w2.label = 'x2w2'
    x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1x2w2'
    n = x1w1x2w2 + b; n.label = 'n'
    o = n.tanh(); o.label = 'o'
    return x1, x2, w1, w2, b, x1w1, x2w2, x1w1x2w2, n, o

x1, x2, w1, w2, b, x1w1, x2w2, x1w1x2w2, n, o = neuron()
# predict: every grad box shows ___ because ___
draw_dot(o)

## Block 1: a `_backward` for each operation (~1:09 to ~1:17, 25 min)
Each node gets a small function that pushes *its* gradient one step back to its children: local derivative × gradient flowing in, the rule you wrote last time.

1. In `__init__`, add `self._backward = lambda: None`. Why a do-nothing function rather than `None`? (Think about which nodes never get a real one.)
2. **Before he does:** write `out._backward` inside `__add__`. It's a closure: define `def _backward():` inside the method, then `out._backward = _backward`. What does it need to reach that isn't passed in as an argument?
3. Then `__mul__` and `tanh` yourself. For `tanh`, use your result from last time, 1 − o².
4. Run it by hand: set `o.grad = 1.0`, then call `o._backward()`, `n._backward()` and so on down the graph. Check against last session's hand values: x1 −1.5, w1 1, x2 0.5, w2 0.

Questions to answer in a comment:
- Why do you have to set `o.grad = 1.0` yourself?
- Call them in a **wrong** order (e.g. `n._backward()` before `o._backward()`). Predict what goes wrong, then try it on a fresh `neuron()`.

In [ ]:
# Paste your updated Value class here (with _backward), then re-run neuron().

In [ ]:
# By hand, in order. predict: x1 ___, w1 ___, x2 ___, w2 ___

In [ ]:
# Wrong order. predict: ___

## Block 2: topological sort → `backward()` (~1:17 to ~1:22, 25 min)
Calling `_backward` by hand only worked because *you* knew the order. Rule: a node's `_backward` may only run once its own `grad` is final, i.e. after every node it feeds into has run.

1. **Before he does:** write `topo(root)`, a list where every node appears *after* all its children. Hint: it's your `trace` DFS from session 1, with one change: append the node *after* visiting its children.
2. Predict the list for the neuron (labels only), then print `[v.label for v in topo(o)]`. Is it the only valid order?
3. Wrap it as a method: `def backward(self):` builds the topo list, sets `self.grad = 1.0`, then calls `_backward` on each node in **reversed** order. Why reversed?
4. Fresh `neuron()`, `o.backward()`, `draw_dot(o)`. All four leaf gradients should match last session.

Think about (interview-style): could this graph ever contain a cycle? Why not, given how `Value` objects are created?

In [ ]:
# topo(root). predict the label order: ___

In [ ]:
# backward() as a method, then:
# x1, x2, w1, w2, b, x1w1, x2w2, x1w1x2w2, n, o = neuron()
# o.backward()
# draw_dot(o)

## Block 3: the bug, before he shows it (~1:22 to ~1:27, 20 min)
Don't watch this part until you've predicted.

```python
a = Value(3.0, label='a')
b = a + a; b.label = 'b'
b.backward()
```
1. On paper: what *should* `a.grad` be? (b = 2a.) Now predict what your code prints. Run it.
2. How many elements are in `b._prev`? Does that matter for `trace`? For `backward`?
3. Explain exactly which line causes it. Then fix it.
4. A harder case, by hand first, then in code:
   ```python
   a = Value(-2.0, label='a'); b = Value(3.0, label='b')
   d = a * b; e = a + b; f = d * e
   f.backward()
   ```
   `a` reaches `f` by two routes. What should `a.grad` be? Check it with your central-difference `numgrad` on a plain-float `f_of(a, b)`.
5. Now that gradients accumulate: call `f.backward()` a **second** time without rebuilding. Predict `a.grad`, then run. (You'll need the answer in session 4.)

In [ ]:
# b = a + a. should be ___, predict the code gives ___

In [ ]:
# two routes. by hand: dF/da = ___, dF/db = ___

In [ ]:
# numgrad check on plain floats

In [ ]:
# backward() twice. predict a.grad = ___

## Block 4 (stretch, only if there's time): break `tanh` into pieces (~1:27 to ~1:39)
Last time you couldn't derive d tanh/dn (needs the quotient rule). Here's another way: build tanh from operations whose derivatives you *do* know, and let `backward()` do the chain rule.

Add `exp`, `__pow__` (int/float powers only), `__neg__`, `__sub__`, `__truediv__` (as `self * other**-1`) and `__rsub__`/`__rtruediv__` if you need them. Then rebuild the neuron with `e = (2*n).exp(); o = (e - 1) / (e + 1)` and check every leaf gradient matches the version with the `tanh` node.

Question: the graph is now much bigger but gives the same gradients. What does that tell you about where to draw the line around an "operation"?

## Reflection (10 min)
Add to `notes/micrograd.md`: **learned / surprised / question it opened**.

CS x Phil: when `a` affects `f` through two routes, its gradient is the *sum* of what flows back along each. Two questions to pick from:
- Causation: is `a` one cause of `f` or two? Statisticians split a total effect into a *direct* and an *indirect* (mediated) effect. Does the sum mean the routes are independent, or is that only true for small nudges? (Remember the 0.72 vs 0.71 cross term.)
- Knowledge: the `=` bug gave a wrong but perfectly plausible number with no error. What caught it? Why is a check that shares no code with the thing it checks worth more than one that does?

Then tick the boxes in `micrograd/README.md` and:
`git add -A && git commit -m "micrograd 3: backward()" && git push`